# 04 — Sentiment Aggregation & JST Join  *(augmented_lags — clean rebuild)*

**This is a complete clean rewrite. It replaces the original notebook 04 and the earlier augmented version.**

What this notebook does:
1. Loads `bis_sentiment_raw.csv` (output of Notebook 03)
2. Maps institution names directly to **3-letter JST ISO codes** (no 2-letter intermediate step)
3. Filters to 18 JST countries, 1997–2020
4. Aggregates to annual country-year sentiment means
5. Fills coverage gaps with per-country means
6. Creates **t−1 AND t−2** sentiment lags
7. Joins to JST macro panel
8. Saves `jst_sentiment_master.csv` (432 rows, with lag1 + lag2 columns)

| Input | File |
|---|---|
| Raw sentiment | `data/processed/bis_sentiment_raw.csv` |
| JST macro | `data/raw/JSTdatasetR6.xlsx` |

| Output | File |
|---|---|
| Annual sentiment | `data/processed/sentiment_annual.csv` |
| Master dataset | `data/processed/jst_sentiment_master.csv` |


## Cell 1 — Paths and imports

In [28]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── YOUR PATH — change only if your OneDrive folder is named differently ──
BASE_DIR = Path(r'C:\Users\Owner\OneDrive\dissertation')
# ─────────────────────────────────────────────────────────────────────────

RAW_DIR       = BASE_DIR / 'data' / 'raw'
PROC_DIR      = BASE_DIR / 'data' / 'processed'
BIS_DIR       = RAW_DIR  / 'Bis_Org_Speaches'

RAW_SENTIMENT = PROC_DIR / 'bis_sentiment_raw.csv'
JST_FILE      = RAW_DIR  / 'JSTdatasetR6.xlsx'
OUT_ANNUAL    = PROC_DIR / 'sentiment_annual.csv'
OUT_MASTER    = PROC_DIR / 'jst_sentiment_master.csv'

PROC_DIR.mkdir(parents=True, exist_ok=True)

print('Input file check:')
for fp in [RAW_SENTIMENT, JST_FILE]:
    tag = '✅  found' if fp.exists() else '❌  MISSING — check path above'
    print(f'  {tag}  →  {fp.name}')
print()
print('Output will be written to:')
print(f'  {OUT_ANNUAL}')
print(f'  {OUT_MASTER}')

# 18 JST country ISO codes — 3-letter throughout this notebook
JST_ISOS = [
    'USA','GBR','DEU','FRA','ITA','ESP','NLD','BEL',
    'PRT','IRL','CHE','JPN','AUS','CAN','SWE','NOR','DNK','FIN'
]
print(f'\nJST countries: {len(JST_ISOS)}')
print(JST_ISOS)

Input file check:
  ✅  found  →  bis_sentiment_raw.csv
  ✅  found  →  JSTdatasetR6.xlsx

Output will be written to:
  C:\Users\Owner\OneDrive\dissertation\data\processed\sentiment_annual.csv
  C:\Users\Owner\OneDrive\dissertation\data\processed\jst_sentiment_master.csv

JST countries: 18
['USA', 'GBR', 'DEU', 'FRA', 'ITA', 'ESP', 'NLD', 'BEL', 'PRT', 'IRL', 'CHE', 'JPN', 'AUS', 'CAN', 'SWE', 'NOR', 'DNK', 'FIN']


## Cell 2 — Load raw sentiment scores

In [30]:
df_raw = pd.read_csv(RAW_SENTIMENT)

print(f'Rows        : {len(df_raw):,}')
print(f'Columns     : {df_raw.columns.tolist()}')
print(f'Year range  : {df_raw["year"].min()} – {df_raw["year"].max()}')
print(f'Missing P_neg : {df_raw["P_neg"].isna().sum()}')
print()
print('First 3 rows:')
print(df_raw.head(3).to_string())
print()
# Confirm the description column exists — this is what we use for ISO mapping
if 'description' in df_raw.columns:
    print('Sample description values:')
    print(df_raw['description'].dropna().head(5).tolist())
else:
    print('WARNING: no description column found — check column names above')

Rows        : 16,622
Columns     : ['url', 'year', 'date', 'author', 'description', 'P_pos', 'P_neg', 'P_neutral']
Year range  : 1997 – 2020
Missing P_neg : 4077

First 3 rows:
                                       url  year                 date              author                                                                                                                                                                                          description     P_pos     P_neg  P_neutral
0  https://www.bis.org/review/r970512a.pdf  1997  1997-04-24 00:00:00    Laurence H Meyer                                               Remarks by Mr. Laurence H. Meyer, a member of the Board of Governors of the US Federal Reserve System, at the Forecasters Club of New York on 24/4/97.  0.132078  0.230044   0.637878
1  https://www.bis.org/review/r970605b.pdf  1997  1997-05-26 00:00:00     Lars Heikensten                                                                Address by the Deputy Governor of 

## Cell 3 — Map institution names to 3-letter JST ISO codes

This is the critical fix vs the original notebook.  
We map **directly to 3-letter codes** (USA, GBR, DEU …) in one step.  
No 2-letter intermediate, no remap needed later.

In [32]:
# ── Complete institution → 3-letter JST ISO mapping ──────────────────────
INSTITUTION_TO_ISO3 = {
    # USA
    'federal reserve':               'USA',
    'board of governors':            'USA',
    'federal open market':           'USA',
    'new york fed':                  'USA',
    'new york federal':              'USA',
    # UK
    'bank of england':               'GBR',
    # Germany
    'deutsche bundesbank':           'DEU',
    'bundesbank':                    'DEU',
    # France
    'banque de france':              'FRA',
    'bank of france':                'FRA',
    # Italy
    'banca d italia':                'ITA',
    "banca d'italia":                'ITA',
    'bank of italy':                 'ITA',
    # Spain
    'banco de espana':               'ESP',
    'banco de españa':               'ESP',
    'bank of spain':                 'ESP',
    # Netherlands
    'nederlandsche bank':            'NLD',
    'netherlands bank':              'NLD',
    'dutch central bank':            'NLD',
    # Belgium
    'national bank of belgium':      'BEL',
    'banque nationale de belgique':  'BEL',
    'belgique':                      'BEL',
    # Portugal
    'banco de portugal':             'PRT',
    'bank of portugal':              'PRT',
    # Ireland
    'central bank of ireland':       'IRL',
    'bank of ireland':               'IRL',
    # Switzerland
    'swiss national bank':           'CHE',
    'schweizerische nationalbank':   'CHE',
    # Japan
    'bank of japan':                 'JPN',
    # Australia
    'reserve bank of australia':     'AUS',
    'bank of australia':             'AUS',
    # Canada
    'bank of canada':                'CAN',
    # Sweden
    'riksbank':                      'SWE',
    'sveriges riksbank':             'SWE',
    'bank of sweden':                'SWE',
    # Norway
    'norges bank':                   'NOR',
    'bank of norway':                'NOR',
    # Denmark
    'danmarks nationalbank':         'DNK',
    'bank of denmark':               'DNK',
    # Finland
    'bank of finland':               'FIN',
    'suomen pankki':                 'FIN',
    # ECB speeches — drop these (not a JST country)
    # We intentionally do NOT map ECB → DEU here
    # ECB speeches will have iso=None and be dropped in Cell 4
}

def map_iso3(description):
    """Map a description string to a 3-letter JST ISO code."""
    if pd.isna(description):
        return None
    desc_lower = str(description).lower()
    for key, iso3 in INSTITUTION_TO_ISO3.items():
        if key in desc_lower:
            return iso3
    return None

# Apply mapping
df_raw['iso'] = df_raw['description'].apply(map_iso3)

# Report
mapped    = df_raw['iso'].notna().sum()
unmapped  = df_raw['iso'].isna().sum()
print(f'Speeches with ISO mapped   : {mapped:,}')
print(f'Speeches without ISO       : {unmapped:,}  (non-JST institutions — will be dropped)')
print()
print('ISO distribution:')
iso_counts = df_raw['iso'].value_counts()
print(iso_counts.to_string())
print()
# Verify all codes are 3-letter and in JST_ISOS
codes_found = df_raw['iso'].dropna().unique()
non_jst = [c for c in codes_found if c not in JST_ISOS]
if non_jst:
    print(f'WARNING: non-JST codes present: {non_jst}')
    print('These will be dropped in Cell 4.')
else:
    print('✅  All mapped codes are valid JST 3-letter ISO codes.')

Speeches with ISO mapped   : 7,970
Speeches without ISO       : 8,652  (non-JST institutions — will be dropped)

ISO distribution:
iso
USA    2138
DEU     759
JPN     662
GBR     655
CAN     503
AUS     482
SWE     469
CHE     370
FRA     341
ITA     304
ESP     286
NOR     262
IRL     215
NLD     170
FIN     153
DNK      87
PRT      69
BEL      45

✅  All mapped codes are valid JST 3-letter ISO codes.


## Cell 4 — Filter to 18 JST countries + 1997–2020 and aggregate annually

In [34]:
# Filter to JST countries and study period
df_jst = df_raw[
    df_raw['iso'].isin(JST_ISOS) &
    df_raw['year'].between(1997, 2020)
].copy()

print(f'Speeches before filter : {len(df_raw):,}')
print(f'Speeches after filter  : {len(df_jst):,}')
print(f'Countries represented  : {df_jst["iso"].nunique()} / 18')
print()

# Check if any of the 18 countries have zero speeches
missing_countries = [iso for iso in JST_ISOS if iso not in df_jst['iso'].values]
if missing_countries:
    print(f'WARNING: no speeches found for: {missing_countries}')
    print('These will be gap-filled with cross-country means in Cell 5.')
else:
    print('✅  All 18 JST countries have at least one speech.')
print()

# Aggregate to annual country-year means
sentiment_annual = (
    df_jst
    .groupby(['year', 'iso'])
    .agg(
        P_pos     =('P_pos',     'mean'),
        P_neg     =('P_neg',     'mean'),
        P_neutral =('P_neutral', 'mean'),
        n_speeches=('P_neg',     'count'),
    )
    .reset_index()
)
sentiment_annual['net_sentiment'] = (
    sentiment_annual['P_pos'] - sentiment_annual['P_neg']
).round(6)
for col in ['P_pos', 'P_neg', 'P_neutral']:
    sentiment_annual[col] = sentiment_annual[col].round(6)

sentiment_annual.to_csv(OUT_ANNUAL, index=False)

print(f'Annual sentiment rows  : {len(sentiment_annual)}  (expect up to 432 = 18 × 24)')
print(f'Saved → {OUT_ANNUAL}')
print()
print(sentiment_annual.head(12).to_string(index=False))

Speeches before filter : 16,622
Speeches after filter  : 7,970
Countries represented  : 18 / 18

✅  All 18 JST countries have at least one speech.

Annual sentiment rows  : 393  (expect up to 432 = 18 × 24)
Saved → C:\Users\Owner\OneDrive\dissertation\data\processed\sentiment_annual.csv

 year iso    P_pos    P_neg  P_neutral  n_speeches  net_sentiment
 1997 AUS 0.152554 0.247528   0.599918           7      -0.094974
 1997 CAN 0.317820 0.271316   0.410865           6       0.046504
 1997 CHE 0.131788 0.137297   0.730915           1      -0.005509
 1997 DEU 0.171991 0.076354   0.751656           8       0.095637
 1997 FIN 0.562373 0.109327   0.328300           1       0.453046
 1997 FRA 0.331707 0.410152   0.258141           7      -0.078445
 1997 GBR 0.209627 0.154243   0.636130           8       0.055384
 1997 IRL 0.169480 0.089639   0.740882           1       0.079841
 1997 ITA 0.130515 0.115714   0.753770           3       0.014801
 1997 JPN 0.237929 0.457307   0.304764          21 

## Cell 5 — Coverage diagnostic

In [36]:
# Build the complete 432-row grid (all 18 countries × all 24 years)
full_grid = pd.MultiIndex.from_product(
    [range(1997, 2021), JST_ISOS], names=['year', 'iso']
).to_frame(index=False)

check   = full_grid.merge(sentiment_annual[['year','iso','P_neg']], on=['year','iso'], how='left')
missing = check[check['P_neg'].isna()]

print(f'Expected country-years  : {len(full_grid)}')
print(f'With sentiment data     : {check["P_neg"].notna().sum()}')
print(f'Missing country-years   : {len(missing)}')

if len(missing) > 0:
    print()
    print('Missing breakdown:')
    miss_summary = (
        missing.groupby('iso')['year']
        .agg(['count','min','max'])
        .rename(columns={'count':'n_missing','min':'first_missing','max':'last_missing'})
    )
    print(miss_summary.to_string())
    print()
    print('These gaps will be filled with per-country means in Cell 6.')
else:
    print('\n✅  Full 432-row coverage — no gaps to fill.')

Expected country-years  : 432
With sentiment data     : 389
Missing country-years   : 43

Missing breakdown:
     n_missing  first_missing  last_missing
iso                                        
BEL          7           1997          2020
DNK          4           1997          2020
ESP          4           1997          2001
FIN          3           1999          2002
FRA          1           1998          1998
IRL          8           1998          2009
ITA          2           2002          2003
NOR          2           1997          1998
PRT         12           1997          2009

These gaps will be filled with per-country means in Cell 6.


## Cell 6 — Fill gaps and build lag1 + lag2 sentiment features

In [38]:
# Merge sentiment onto the complete 432-row grid
sa = full_grid.merge(sentiment_annual, on=['year','iso'], how='left')

print(f'Rows after merge : {len(sa)}  (should be exactly 432)')

# Fill missing country-years with that country's mean
SENT_COLS = ['P_pos', 'P_neg', 'P_neutral', 'net_sentiment']
for col in SENT_COLS:
    country_mean = sa.groupby('iso')[col].transform('mean')
    n_filled = sa[col].isna().sum()
    sa[col] = sa[col].fillna(country_mean)
    if n_filled > 0:
        print(f'  {col}: filled {n_filled} gaps with country mean')

# CRITICAL: sort by [iso, year] before any shift()
sa = sa.sort_values(['iso', 'year']).reset_index(drop=True)

# ── Create t−1 and t−2 lags ──────────────────────────────────────────────
for col in SENT_COLS:
    sa[f'{col}_lag1'] = sa.groupby('iso')[col].shift(1)   # t−1
    sa[f'{col}_lag2'] = sa.groupby('iso')[col].shift(2)   # t−2  (NEW)

lag1_cols = [c for c in sa.columns if c.endswith('_lag1')]
lag2_cols = [c for c in sa.columns if c.endswith('_lag2')]

print()
print(f't−1 lag columns: {lag1_cols}')
print(f't−2 lag columns: {lag2_cols}')
print()
print(f'NaN in lag1 (expect 18 — one per country for 1997) : {sa[lag1_cols[0]].isna().sum()}')
print(f'NaN in lag2 (expect 36 — 1997 and 1998 per country): {sa[lag2_cols[0]].isna().sum()}')
print()

# Verify alignment for USA
print('USA alignment check (should show NaN for 1997, real values from 1998 for lag1):')
usa = sa[sa['iso']=='USA'][['year','P_neg','P_neg_lag1','P_neg_lag2']].head(6)
print(usa.to_string(index=False))

Rows after merge : 432  (should be exactly 432)
  P_pos: filled 43 gaps with country mean
  P_neg: filled 43 gaps with country mean
  P_neutral: filled 43 gaps with country mean
  net_sentiment: filled 43 gaps with country mean

t−1 lag columns: ['P_pos_lag1', 'P_neg_lag1', 'P_neutral_lag1', 'net_sentiment_lag1']
t−2 lag columns: ['P_pos_lag2', 'P_neg_lag2', 'P_neutral_lag2', 'net_sentiment_lag2']

NaN in lag1 (expect 18 — one per country for 1997) : 18
NaN in lag2 (expect 36 — 1997 and 1998 per country): 36

USA alignment check (should show NaN for 1997, real values from 1998 for lag1):
 year    P_neg  P_neg_lag1  P_neg_lag2
 1997 0.163145         NaN         NaN
 1998 0.163982    0.163145         NaN
 1999 0.149645    0.163982    0.163145
 2000 0.158812    0.149645    0.163982
 2001 0.196379    0.158812    0.149645
 2002 0.209385    0.196379    0.158812


## Cell 7 — Join to JST macro panel

In [40]:
# Load JST R6
jst = pd.read_excel(JST_FILE)
jst.columns = jst.columns.str.lower().str.strip()

print(f'JST total rows   : {len(jst):,}')
print(f'JST columns      : {jst.columns.tolist()}')
print()

# Filter to study window
jst_window = jst[jst['year'].between(1997, 2020)].copy()
print(f'JST rows 1997-2020 : {len(jst_window)}')

# Check iso column in JST
if 'iso' in jst_window.columns:
    print(f'JST ISO codes      : {sorted(jst_window["iso"].unique())}')
else:
    print('WARNING: no iso column in JST — check column names')
print()

# Merge on (year, iso)
df_master = jst_window.merge(sa, on=['year','iso'], how='left')

print(f'Master rows        : {len(df_master)}  (expect 432)')
print(f'Master columns     : {len(df_master.columns)}')
print(f'P_neg_lag1 present : {df_master["P_neg_lag1"].notna().sum()} / {len(df_master)}')
print(f'P_neg_lag2 present : {df_master["P_neg_lag2"].notna().sum()} / {len(df_master)}')
print()

# Save
df_master.to_csv(OUT_MASTER, index=False)
print(f'✅  Saved → {OUT_MASTER}')

# Key column check
key_cols = ['year','iso','crisisjst','tloans','ltrate','stir','debtgdp','hpnom',
            'P_neg_lag1','P_pos_lag1','net_sentiment_lag1',
            'P_neg_lag2','P_pos_lag2','net_sentiment_lag2']
found   = [c for c in key_cols if c in df_master.columns]
missing = [c for c in key_cols if c not in df_master.columns]
print()
print(f'Key columns found   : {found}')
if missing:
    print(f'Key columns missing : {missing}  ← check JST column names')

JST total rows   : 2,718
JST columns      : ['year', 'country', 'iso', 'ifs', 'pop', 'rgdpmad', 'rgdpbarro', 'rconsbarro', 'gdp', 'iy', 'cpi', 'ca', 'imports', 'exports', 'narrowm', 'money', 'stir', 'ltrate', 'hpnom', 'unemp', 'wage', 'debtgdp', 'revenue', 'expenditure', 'xrusd', 'tloans', 'tmort', 'thh', 'tbus', 'bdebt', 'lev', 'ltd', 'noncore', 'crisisjst', 'crisisjst_old', 'peg', 'peg_strict', 'peg_type', 'peg_base', 'jsttrilemmaiv', 'eq_tr', 'housing_tr', 'bond_tr', 'bill_rate', 'rent_ipolated', 'housing_capgain_ipolated', 'housing_capgain', 'housing_rent_rtn', 'housing_rent_yd', 'eq_capgain', 'eq_dp', 'eq_capgain_interp', 'eq_tr_interp', 'eq_dp_interp', 'bond_rate', 'eq_div_rtn', 'capital_tr', 'risky_tr', 'safe_tr']

JST rows 1997-2020 : 432
JST ISO codes      : ['AUS', 'BEL', 'CAN', 'CHE', 'DEU', 'DNK', 'ESP', 'FIN', 'FRA', 'GBR', 'IRL', 'ITA', 'JPN', 'NLD', 'NOR', 'PRT', 'SWE', 'USA']

Master rows        : 432  (expect 432)
Master columns     : 72
P_neg_lag1 present : 414 / 432


## Cell 8 — Final validation

In [42]:
print('=' * 60)
print('  MASTER DATASET VALIDATION (augmented_lags)')
print('=' * 60)

checks = [
    ('Total rows = 432',            len(df_master) == 432),
    ('Countries = 18',              df_master['iso'].nunique() == 18),
    ('Year range 1997-2020',        df_master['year'].min() == 1997
                                    and df_master['year'].max() == 2020),
    ('P_neg_lag1 present',          'P_neg_lag1' in df_master.columns),
    ('P_neg_lag2 present',          'P_neg_lag2' in df_master.columns),
    ('net_sentiment_lag1 present',  'net_sentiment_lag1' in df_master.columns),
    ('net_sentiment_lag2 present',  'net_sentiment_lag2' in df_master.columns),
    ('P_neg_lag1 has values',       df_master['P_neg_lag1'].notna().sum() > 380),
    ('P_neg_lag2 has values',       df_master['P_neg_lag2'].notna().sum() > 360),
    ('crisisjst present',           any('crisis' in c.lower() for c in df_master.columns)),
    ('tloans present',              any('tloans' in c.lower() for c in df_master.columns)),
]

all_ok = True
for label, result in checks:
    icon = '✅' if result else '❌'
    print(f'  {icon}  {label}')
    if not result:
        all_ok = False

print()
if all_ok:
    print('  ✅  All checks passed.')
    print('  Ready to run 05_Model_Training_augmented_lags.ipynb')
else:
    print('  ❌  Some checks failed — do not proceed to notebook 05.')

print()
print('Lag column NaN counts (for reference):')
for c in ['P_neg_lag1','P_neg_lag2']:
    if c in df_master.columns:
        print(f'  {c}: {df_master[c].isna().sum()} NaN  '
              f'({df_master[c].notna().sum()} valid values)')
print()
print(f'Output file: {OUT_MASTER}')
print(f'File size  : {OUT_MASTER.stat().st_size // 1024} KB')

  MASTER DATASET VALIDATION (augmented_lags)
  ✅  Total rows = 432
  ✅  Countries = 18
  ✅  Year range 1997-2020
  ✅  P_neg_lag1 present
  ✅  P_neg_lag2 present
  ✅  net_sentiment_lag1 present
  ✅  net_sentiment_lag2 present
  ✅  P_neg_lag1 has values
  ✅  P_neg_lag2 has values
  ✅  crisisjst present
  ✅  tloans present

  ✅  All checks passed.
  Ready to run 05_Model_Training_augmented_lags.ipynb

Lag column NaN counts (for reference):
  P_neg_lag1: 18 NaN  (414 valid values)
  P_neg_lag2: 36 NaN  (396 valid values)

Output file: C:\Users\Owner\OneDrive\dissertation\data\processed\jst_sentiment_master.csv
File size  : 350 KB
